In [17]:
from dotenv import load_dotenv
load_dotenv()

True

## EMBEDDING FUNCTION

In [18]:
from langchain_openai import OpenAIEmbeddings

embedding=OpenAIEmbeddings(
    model="text-embedding-3-small"
)



## FILE LOADING 

In [22]:
import os
os.chdir("c:/Users/moham/Videos/farmer_bot/")
print(os.getcwd())


c:\Users\moham\Videos\farmer_bot


In [24]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("weed_guide.pdf")
docs = loader.load()
len(docs)


6

## CHUNKING 

In [25]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=500
)

chunks = splitter.split_documents(docs)
len(chunks)

18

## CHROMA SETUP


In [26]:
from langchain_community.vectorstores import Chroma


vector_store=Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./db"
)


## PROMPT TEMPLATE 

In [27]:
from langchain_core.prompts import PromptTemplate

prompt_template="""

You are an expert weed identification assistant.

Your task is to answer questions using ONLY the provided context.

====================
SOURCE OF TRUTH
===============

The provided context is the ONLY source of truth.

You MUST:

* Use only information found in the context.
* Never use external knowledge.
* Never use prior knowledge.
* Never guess.
* Never invent facts.
* Never assume missing information.

If a fact is not supported by the context, treat it as unknown.

====================
LANGUAGE RULES
==============

* Answer in the SAME language as the user's question.
* Do not translate unless explicitly requested.
* Preserve scientific names exactly as they appear.

====================
REASONING RULES
===============

You may:

* Compare information across multiple context sections.
* Combine information from multiple context chunks.
* Perform filtering.
* Perform elimination.
* Perform logical reasoning based only on the provided context.

You must reason internally.

Do NOT reveal:

* Chain of thought
* Internal reasoning
* Analysis
* Candidate evaluation
* Elimination steps
* Relevant facts section

Return only the final answer.

====================
MULTI-CONDITION QUESTIONS
=========================

When a question contains multiple conditions:

Examples:

* and
* both
* all of the following
* satisfies every condition
* matches all conditions

Treat the conditions as a logical AND.

A weed MUST satisfy EVERY condition to be included.

Example:

Question:
"Which weeds are biennial and have purple flowers?"

Correct:
Biennial AND Purple flowers

Incorrect:
Biennial OR Purple flowers

If a weed fails even ONE condition, exclude it.

====================
OR CONDITIONS
=============

Only use OR logic when the user explicitly requests OR logic.

Example:

Question:
"Which weeds are biennial or have purple flowers?"

Correct:
Include weeds satisfying either condition.

====================
FILTERING PROCESS
=================

For every question:

1. Extract all conditions.
2. Identify candidate weeds from the context.
3. Check every candidate against every condition.
4. Exclude candidates that fail any condition.
5. Include all candidates that satisfy all required conditions.
6. Verify no valid candidate is omitted.
7. Verify no invalid candidate is included.

====================
FINAL VALIDATION
================

Before producing the answer, silently verify:

□ Every answer appears in the context.
□ Every answer satisfies all required conditions.
□ No answer violates any condition.
□ No valid candidate has been omitted.
□ The answer language matches the user's language.
□ Every statement is supported by context.

====================
INSUFFICIENT INFORMATION
========================

If the answer cannot be determined from the context after careful verification, respond EXACTLY:

I don't know the answer based on the provided context.

====================
CONTEXT
=======

{context}

====================
QUESTION
========

{question}

====================
ANSWER
======

"""

prompt = PromptTemplate.from_template(prompt_template)

## LANGCHAIN OPENAI SETUP

In [28]:
from langchain_openai import ChatOpenAI

llm=ChatOpenAI(
    model="gpt-5.5",
    
)



In [29]:
from langchain_classic.vectorstores import Chroma

vector_retreival=Chroma(
    persist_directory="./db",
    embedding_function=embedding
)

## Chaining


In [30]:
from langchain_core.runnables import RunnableLambda
def create_context(question):

    retriever = vector_retreival.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 15,
        "fetch_k": 40
    }
)
    docs = retriever.invoke(question)

    context = "\n\n".join(
        f"Document {i+1} (Page {doc.metadata.get('page', 'Unknown')}):\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

    return {
        "question": question,
        "context": context
    }
context_maker=RunnableLambda(create_context)


In [31]:
from langchain_classic.vectorstores import Chroma

vector_retreival=Chroma(
    persist_directory="./db",
    embedding_function=embedding
)

In [32]:
from langchain_core.tools import tool
from langdetect import detect

@tool
def detect_language(text: str) -> str:
    """Detect user's language."""
    return detect(text)

In [33]:
@tool
def translate_for_retrieval(text: str) -> str:
    """Convert query into English for retrieval."""
    
    response = llm.invoke(
        f"""
        Translate this query into English.
        Return only the translated query.

        Query:
        {text}
        """
    )

    return response.content

In [34]:
@tool
def translate_answer(
    text: str,
    language: str
) -> str:
    """Translate answer into target language."""

    response = llm.invoke(
        f"""
        Translate the following answer into {language}.

        Answer:
        {text}
        """
    )

    return response.content

In [35]:
tools = [
    detect_language,
    translate_for_retrieval,
    translate_answer
    
]

llm_with_tools = llm.bind_tools(tools)

In [36]:
chain=context_maker|prompt|llm_with_tools
user_input = input("Enter: ")
final_ans = chain.invoke(user_input)

print(final_ans.content)

Musk thistle is **Carduus nulans**.

It is a **biennial, or sometimes winter annual forb**. Key identification features include:

- **Broad, spine-tipped bracts** under the flower
- **Flowering heads are terminal, solitary, and usually nodding**
- Flowers are **deep rose, violet or purple**, occasionally white
- Mature plants can grow **up to 6 feet tall**
- Leaves are **alternate, dark green, deeply lobed, and spiny margined**
- It has a **fleshy taproot**


In [37]:
collection = vector_retreival._collection

print(collection.count())

18
